In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from pathlib import Path
from joblib import Parallel, delayed

from pd_estim_A.data.data_import import (
    load_data, load_ecb_1y_yield,
    fill_liabilities, drop_high_leverage_firms,
    prepare_merton_inputs,
)
from pd_estim_A.models.merton.bayesian_merton import (
    process_one_firm_bayesian_merton,
)
from pd_estim_A.data.cds_df import get_cds_panel

In [2]:
# Data processing and df preparation
print(Path.cwd())

data_path = Path.cwd() / ".." / "data" / "raw"
output_path = Path.cwd() / ".." / "data" / "derived"

# raw accounting / equity data
ret_daily, bs, coverage = load_data(
    data_path / "Jan2025_Accenture_Dataset_ErasmusCase.xlsx",
    start_date="2012-01-01",
    end_date="2025-12-19",
    enforce_coverage=True,
    coverage_tol=0.995,
    liabilities_scale="auto",
    verbose=True,
)

# risk-free rate
df_rf = load_ecb_1y_yield(
    startPeriod="2010-01-01",
    endPeriod="2025-12-31",
    out_file=output_path / "ecb_yc_1y_aaa.xml",
    verify_ssl=True,
)

# calendar and debt interpolation
df_cal = ret_daily[["date"]].drop_duplicates().sort_values("date").reset_index(drop=True)
debt_daily = fill_liabilities(bs, df_cal)

# leverage filter
ret_filt, bs_filt, lev_by_firm, dropped = drop_high_leverage_firms(
    ret_daily,
    bs,
    df_calendar=df_cal,
    debt_daily=debt_daily,
    lev_threshold=8.0,
    lev_agg="median",
    verbose=True,
)

# keep debt panel aligned with surviving firms
keep = set(ret_filt["gvkey"].astype(str).unique())
debt_daily_filt = debt_daily[debt_daily["gvkey"].astype(str).isin(keep)].copy()

# structural-model input panel
merton_df = prepare_merton_inputs(
    ret_filt,
    bs_filt,
    df_rf,
    debt_daily=debt_daily_filt,
)

# CDS panel
cds = get_cds_panel(
    project_root=Path.cwd() / "..",
    save_csv=False,
    verbose=True,
)

# harmonize types
merton = merton_df.copy()
merton["gvkey"] = merton["gvkey"].astype(str)
merton["date"] = pd.to_datetime(merton["date"])

cds["gvkey"] = cds["gvkey"].astype(str)
cds["date"] = pd.to_datetime(cds["date"])

# keep only firms that appear in both datasets
common_gv = sorted(set(merton["gvkey"].unique()) & set(cds["gvkey"].unique()))
merton = merton[merton["gvkey"].isin(common_gv)].copy()
cds = cds[cds["gvkey"].isin(common_gv)].copy()

# restrict CDS to Merton panel date range
dmin, dmax = merton["date"].min(), merton["date"].max()
cds = cds[(cds["date"] >= dmin) & (cds["date"] <= dmax)].copy()

# merge CDS backward onto structural panel
merton = merton.sort_values(["date", "gvkey"]).reset_index(drop=True)
cds = cds.sort_values(["date", "gvkey"]).reset_index(drop=True)

merged_cds = pd.merge_asof(
    merton,
    cds,
    on="date",
    by="gvkey",
    direction="backward",
    allow_exact_matches=True,
)

# drop rows still missing CDS
merton_df = merged_cds.dropna(subset=["cds"]).reset_index(drop=True)

print("firms after intersection:", merton_df["gvkey"].nunique())
print("rows after merge:", len(merton_df))
print("date range:", merton_df["date"].min(), "→", merton_df["date"].max())

/Users/telmosantos/Desktop/Merton_NIGbayesian/notebooks_test
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
Data has been written to /Users/telmosantos/Desktop/Merton_NIGbayesian/notebooks_test/../data/derived/ecb_yc_1y_aaa.xml
[drop_high_leverage_firms] agg=median, threshold=8.0
[drop_high_leverage_firms] firms before: 46 | after: 36
[drop_high_leverage_firms] dropped firms: 10
[load_data] Firms (ret_daily): 46
[load_data] Date range (ret_daily): 2012-01-03 .. 2025-12-19
[load_data] Coverage min/median/max: 0.999 / 1.000 / 1.000
[load_data] liabilities_scale_used: 1e+06
[load_data] QA mcap_reported<=0 rows (raw windowed mkt): 62
[get_cds_panel] sheets read: 22 | rows parsed: 67015 | unmapped sheets: 0
firms after intersection: 21
rows after merge: 65569
date range: 2014-01-01 00:

In [3]:
# load precomputed classical Merton results for Bayesian initialization
# this file must contain, at minimum:
# gvkey, train_end_date, training_end, mu_hat, sigma_hat, V_0

classical_init_df = pd.read_csv(output_path / "merton_weekly.csv")

classical_init_df["gvkey"] = classical_init_df["gvkey"].astype(str)
classical_init_df["date"] = pd.to_datetime(classical_init_df["date"], errors="coerce")

if "train_end_date" in classical_init_df.columns:
    classical_init_df["train_end_date"] = pd.to_datetime(
        classical_init_df["train_end_date"],
        errors="coerce",
    )

if "training_end" in classical_init_df.columns:
    classical_init_df["training_end"] = (
        pd.to_numeric(classical_init_df["training_end"], errors="coerce")
        .fillna(0)
        .astype(int)
    )

for c in ["mu_hat", "sigma_hat", "V_0", "V_used", "B_used", "PD_Q", "PD_P"]:
    if c in classical_init_df.columns:
        classical_init_df[c] = pd.to_numeric(classical_init_df[c], errors="coerce")

print("classical init rows:", len(classical_init_df))
print("classical init firms:", classical_init_df["gvkey"].nunique())

if "training_end" in classical_init_df.columns:
    tmp = classical_init_df.loc[classical_init_df["training_end"] == 1].copy()
    print("training_end == 1 rows:", len(tmp))
    print("unique firm-window endpoints:", tmp[["gvkey", "train_end_date"]].drop_duplicates().shape[0])

classical init rows: 10227
classical init firms: 21
training_end == 1 rows: 756
unique firm-window endpoints: 756


In [4]:
# Rolling configuration
TRAIN_YEARS = 2
STEP_FREQ = "QE"
WEEK_ENDING = "W-FRI"
T_HORIZON = 1.0
DATA_END = pd.Timestamp("2024-12-31")
MIN_DAILY_ROWS = 10

LAST_TRAIN_END = DATA_END - pd.offsets.QuarterEnd(1)

# debug / runtime controls
MAX_FIRMS = None
MAX_WINDOWS = None

# Bayesian sampling controls
TARGET_ACCEPT = 0.95
MAX_TREEDEPTH = 15
SEED = 123
HDI_PROB = 0.95
STORE_OOS_DRAWS = False

In [7]:
# one-time preprocessing for speed
panel = merton_df.copy()
panel["gvkey"] = panel["gvkey"].astype(str)
panel["date"] = pd.to_datetime(panel["date"])

needed_cols = ["gvkey", "date", "company", "E", "B", "r", "sigma_E"]
panel = panel[[c for c in needed_cols if c in panel.columns]].copy()

for c in ["E", "B", "r", "sigma_E"]:
    if c in panel.columns:
        panel[c] = pd.to_numeric(panel[c], errors="coerce")

panel["T"] = float(T_HORIZON)

panel = (
    panel.dropna(subset=["date", "E", "B", "r", "T"])
         .query("E > 0 and B > 0")
         .sort_values(["gvkey", "date"])
)

# build per-firm daily panels once
firm_daily = {}
for gvkey, g in panel.groupby("gvkey", sort=False):
    g = g.sort_values("date")
    g = g.groupby("date", as_index=False).last()
    firm_daily[gvkey] = g.set_index("date")

gvkeys_all = sorted(firm_daily.keys())
if MAX_FIRMS is not None:
    gvkeys_all = gvkeys_all[:int(MAX_FIRMS)]

print("Firms loaded:", len(firm_daily), "| Firms in run:", len(gvkeys_all))
print("Panel date range:", panel["date"].min().date(), "to", panel["date"].max().date())
print("LAST_TRAIN_END:", LAST_TRAIN_END.date(), "| DATA_END:", DATA_END.date())

Firms loaded: 21 | Firms in run: 21
Panel date range: 2014-01-01 to 2025-12-19
LAST_TRAIN_END: 2024-09-30 | DATA_END: 2024-12-31


In [8]:
# build the rolling quarter schedule
global_min_date = panel["date"].min()

earliest_end = global_min_date + pd.DateOffset(years=TRAIN_YEARS) - pd.Timedelta(days=1)

train_ends = pd.date_range(start=earliest_end, end=LAST_TRAIN_END, freq=STEP_FREQ)
train_ends = pd.to_datetime(train_ends)

if MAX_WINDOWS is not None:
    train_ends = train_ends[:int(MAX_WINDOWS)]

windows = []
for train_end in train_ends:
    train_start = train_end - pd.DateOffset(years=TRAIN_YEARS) + pd.Timedelta(days=1)
    oos_start = train_end + pd.Timedelta(days=1)
    oos_end = train_end + pd.offsets.QuarterEnd(1)

    windows.append({
        "train_start": pd.Timestamp(train_start),
        "train_end": pd.Timestamp(train_end),
        "oos_start": pd.Timestamp(oos_start),
        "oos_end": pd.Timestamp(oos_end),
    })

windows_df = pd.DataFrame(windows)

print("Number of windows:", len(windows_df))
display(windows_df.head())

Number of windows: 36


,train_start,train_end,oos_start,oos_end
0,2014-01-01,2015-12-31,2016-01-01,2016-03-31
1,2014-04-01,2016-03-31,2016-04-01,2016-06-30
2,2014-07-01,2016-06-30,2016-07-01,2016-09-30
3,2014-10-01,2016-09-30,2016-10-01,2016-12-31
4,2015-01-01,2016-12-31,2017-01-01,2017-03-31


In [10]:
import os

In [17]:
# ---------------- FULL RUN CONFIG ----------------
# all firms / all windows
RUN_GVKEYS = list(gvkeys_all)
RUN_WINDOWS = list(windows)

# Bayesian sampling controls chosen by you
DRAWS = 500
TUNE = 200
CHAINS = 4
CORES = 1                 # keep 1 inside pm.sample; parallelism is across firms via joblib
TARGET_ACCEPT = 0.95
MAX_TREEDEPTH = 15
SEED = 123
HDI_PROB = 0.95
STORE_OOS_DRAWS = True    # this is the largest output table

# use all available CPU cores across firms
N_JOBS = -1
JOBLIB_BACKEND = "loky"
JOBLIB_VERBOSE = 10

# fixed-delta setting
DELTA0 = 0.01

# file naming
RUN_TAG = f"bayes_merton_fixdelta_allfirms_allwindows_c{CHAINS}_tune{TUNE}_draws{DRAWS}"
RUN_DIR = output_path / RUN_TAG
RUN_DIR.mkdir(parents=True, exist_ok=True)

# reduce oversubscription in worker processes
for var in [
    "OMP_NUM_THREADS",
    "OPENBLAS_NUM_THREADS",
    "MKL_NUM_THREADS",
    "VECLIB_MAXIMUM_THREADS",
    "NUMEXPR_NUM_THREADS",
]:
    os.environ[var] = "1"

print("RUN_TAG:", RUN_TAG)
print("RUN_DIR:", RUN_DIR)
print("Firms to run:", len(RUN_GVKEYS))
print("Windows to run:", len(RUN_WINDOWS))
print("Sampling setup:", {
    "draws": DRAWS,
    "tune": TUNE,
    "chains": CHAINS,
    "cores_inside_pymc": CORES,
    "target_accept": TARGET_ACCEPT,
    "max_treedepth": MAX_TREEDEPTH,
    "delta0": DELTA0,
    "store_oos_draws": STORE_OOS_DRAWS,
    "joblib_n_jobs": N_JOBS,
    "joblib_backend": JOBLIB_BACKEND,
})

RUN_TAG: bayes_merton_fixdelta_allfirms_allwindows_c4_tune200_draws500
RUN_DIR: /Users/telmosantos/Desktop/Merton_NIGbayesian/notebooks_test/../data/derived/bayes_merton_fixdelta_allfirms_allwindows_c4_tune200_draws500
Firms to run: 21
Windows to run: 36
Sampling setup: {'draws': 500, 'tune': 200, 'chains': 4, 'cores_inside_pymc': 1, 'target_accept': 0.95, 'max_treedepth': 15, 'delta0': 0.01, 'store_oos_draws': True, 'joblib_n_jobs': -1, 'joblib_backend': 'loky'}


In [18]:
def _run_one_firm_bayesian_merton_full(gvkey: str, firm_idx: int):
    """
    Run the full rolling Bayesian Merton workflow for one firm.
    Parallelization is across firms.
    """
    import time
    import pandas as pd
    import pd_estim_A.models.merton.bayesian_merton as bayes_merton

    t0 = time.time()

    g = firm_daily[gvkey].copy()

    company = None
    if "company" in g.columns and g["company"].notna().any():
        company = g["company"].dropna().iloc[0]

    firm_seed = int(SEED + 100000 * firm_idx)

    try:
        window_summary_df, param_draws_all_df, oos_summary_all_df, oos_draws_all_df = (
            bayes_merton.process_one_firm_bayesian_merton(
                g,
                windows=RUN_WINDOWS,
                classical_init_df=classical_init_df,
                gvkey=gvkey,
                date_col="date",
                week_ending=WEEK_ENDING,
                ann_factor=52.0,
                T_horizon=T_HORIZON,
                min_daily_rows=MIN_DAILY_ROWS,
                min_weekly_obs=104,
                min_weekly_returns=2,
                E_col="E",
                B_col="B",
                r_col="r",
                T_col="T",
                B_scale=1.0,
                sigmaE_col="sigma_E",
                delta0=DELTA0,
                draws=DRAWS,
                tune=TUNE,
                chains=CHAINS,
                cores=CORES,
                target_accept=TARGET_ACCEPT,
                max_treedepth=MAX_TREEDEPTH,
                seed=firm_seed,
                hdi_prob=HDI_PROB,
                store_oos_draws=STORE_OOS_DRAWS,
                classical_gvkey_col="gvkey",
                classical_train_end_col="train_end_date",
                classical_training_end_col="training_end",
                classical_mu_col="mu_hat",
                classical_sigma_col="sigma_hat",
                classical_V0_col="V_0",
            )
        )

        elapsed = time.time() - t0

        if not window_summary_df.empty and "company" not in window_summary_df.columns:
            window_summary_df = window_summary_df.copy()
            window_summary_df["company"] = company

        if not param_draws_all_df.empty and "company" not in param_draws_all_df.columns:
            param_draws_all_df = param_draws_all_df.copy()
            param_draws_all_df["company"] = company

        if not oos_summary_all_df.empty and "company" not in oos_summary_all_df.columns:
            oos_summary_all_df = oos_summary_all_df.copy()
            oos_summary_all_df["company"] = company

        if not oos_draws_all_df.empty and "company" not in oos_draws_all_df.columns:
            oos_draws_all_df = oos_draws_all_df.copy()
            oos_draws_all_df["company"] = company

        firm_status = {
            "gvkey": gvkey,
            "company": company,
            "ok": True,
            "elapsed_sec": elapsed,
            "n_windows_attempted": len(RUN_WINDOWS),
            "n_window_rows": len(window_summary_df),
            "n_param_draw_rows": len(param_draws_all_df),
            "n_oos_summary_rows": len(oos_summary_all_df),
            "n_oos_draw_rows": len(oos_draws_all_df),
            "msg": "ok",
        }

        return {
            "firm_status": pd.DataFrame([firm_status]),
            "window_summary": window_summary_df,
            "param_draws": param_draws_all_df,
            "oos_summary": oos_summary_all_df,
            "oos_draws": oos_draws_all_df,
        }

    except Exception as e:
        elapsed = time.time() - t0

        firm_status = {
            "gvkey": gvkey,
            "company": company,
            "ok": False,
            "elapsed_sec": elapsed,
            "n_windows_attempted": len(RUN_WINDOWS),
            "n_window_rows": 0,
            "n_param_draw_rows": 0,
            "n_oos_summary_rows": 0,
            "n_oos_draw_rows": 0,
            "msg": f"{type(e).__name__}: {str(e)[:500]}",
        }

        return {
            "firm_status": pd.DataFrame([firm_status]),
            "window_summary": pd.DataFrame(),
            "param_draws": pd.DataFrame(),
            "oos_summary": pd.DataFrame(),
            "oos_draws": pd.DataFrame(),
        }

In [19]:
import time

In [20]:
run_start = time.time()

results = Parallel(
    n_jobs=N_JOBS,
    backend=JOBLIB_BACKEND,
    verbose=JOBLIB_VERBOSE,
)(
    delayed(_run_one_firm_bayesian_merton_full)(gvkey, i)
    for i, gvkey in enumerate(RUN_GVKEYS)
)

run_elapsed = time.time() - run_start
print(f"Full run finished in {run_elapsed/60:.2f} minutes.")
print("Number of firm results returned:", len(results))

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 8 concurrent workers.
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Initializing NUTS using adapt_diag...
Sequential sampling (4 chains in 1 job)
NUTS: [mu, sigma, eps]


                                                                                
                              Step      Grad      Sampling                      
  Progre…   Draws   Diverg…   size      evals     Speed      Elapsed   Remain…  
 ────────────────────────────────────────────────────────────────────────────── 
                                                                                
                              Step      Grad      Sampli…                       
  Progre…   Draws   Diverg…   size      evals     Speed     Elapsed   Remaini…  
 ────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━   0       0         0.000     0         0.00      0:00:00   -:--:--   
                                                  draws/s                       
                                                                                
                              Step      Grad      Sampli…                       
  Progre…   Draws   Diverg… 

Sequential sampling (4 chains in 1 job)
NUTS: [mu, sigma, eps]


                                                                                
                              Step      Grad      Sampli…                       
  Progre…   Draws   Diverg…   size      evals     Speed     Elapsed   Remaini…  
 ────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━   14      0         0.000     255       8.53      0:00:02   0:01:26   
                                                  drawss…                       
  ━━━━━━━   0       0         0.000     0         0.00      0:00:02   -:--:--   
                                                  draws/s                       
  ━━━━━━━   0       0         0.000     0         0.00      0:00:02   -:--:--   
                                                  draws/s                       
  ━━━━━━━   0       0         0.000     0         0.00      0:00:02   -:--:--   
                                                  draws/s                       
                            

Sequential sampling (4 chains in 1 job)
NUTS: [mu, sigma, eps]


...

Sequential sampling (4 chains in 1 job)
NUTS: [mu, sigma, eps]


                                                                                
                              Step      Grad      Sampling                      
  Progre…   Draws   Diverg…   size      evals     Speed      Elapsed   Remain…  
 ────────────────────────────────────────────────────────────────────────────── 
                                                                                
                              Step      Grad      Sampli…                       
  Progre…   Draws   Diverg…   size      evals     Speed     Elapsed   Remaini…  
 ────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━   0       0         0.000     0         0.00      0:00:00   -:--:--   
                                                  draws/s                       
                                                                                
                              Step      Grad      Sampli…                       
  Progre…   Draws   Diverg… 

Sequential sampling (4 chains in 1 job)
NUTS: [mu, sigma, eps]


                                                                                
                              Step      Grad      Sampli…                       
  Progre…   Draws   Diverg…   size      evals     Speed     Elapsed   Remaini…  
 ────────────────────────────────────────────────────────────────────────────── 
  ━━━━━━━   28      0         0.000     255       2.70      0:00:11   0:04:16   
                                                  drawss…                       
  ━━━━━━━   0       0         0.000     0         0.00      0:00:11   -:--:--   
                                                  draws/s                       
  ━━━━━━━   0       0         0.000     0         0.00      0:00:11   -:--:--   
                                                  draws/s                       
  ━━━━━━━   0       0         0.000     0         0.00      0:00:11   -:--:--   
                                                  draws/s                       
                            

KeyboardInterrupt: 

In [16]:
firm_status_parts = []
window_summary_parts = []
param_draws_parts = []
oos_summary_parts = []
oos_draws_parts = []

for res in results:
    fs = res.get("firm_status", pd.DataFrame())
    ws = res.get("window_summary", pd.DataFrame())
    pdraw = res.get("param_draws", pd.DataFrame())
    ooss = res.get("oos_summary", pd.DataFrame())
    oosd = res.get("oos_draws", pd.DataFrame())

    if fs is not None and not fs.empty:
        firm_status_parts.append(fs)

    if ws is not None and not ws.empty:
        window_summary_parts.append(ws)

    if pdraw is not None and not pdraw.empty:
        param_draws_parts.append(pdraw)

    if ooss is not None and not ooss.empty:
        oos_summary_parts.append(ooss)

    if oosd is not None and not oosd.empty:
        oos_draws_parts.append(oosd)

firm_status_df = (
    pd.concat(firm_status_parts, ignore_index=True)
    .sort_values(["ok", "gvkey"], ascending=[False, True])
    .reset_index(drop=True)
    if len(firm_status_parts) else pd.DataFrame()
)

window_summary_all_df = (
    pd.concat(window_summary_parts, ignore_index=True)
    .sort_values(["train_end", "gvkey"])
    .reset_index(drop=True)
    if len(window_summary_parts) else pd.DataFrame()
)

param_draws_all_df = (
    pd.concat(param_draws_parts, ignore_index=True)
    .sort_values(["train_end", "gvkey", "sample_id"])
    .reset_index(drop=True)
    if len(param_draws_parts) else pd.DataFrame()
)

oos_summary_all_df = (
    pd.concat(oos_summary_parts, ignore_index=True)
    .sort_values(["train_end", "gvkey", "date"])
    .reset_index(drop=True)
    if len(oos_summary_parts) else pd.DataFrame()
)

oos_draws_all_df = (
    pd.concat(oos_draws_parts, ignore_index=True)
    .sort_values(["train_end", "gvkey", "date", "sample_id"])
    .reset_index(drop=True)
    if len(oos_draws_parts) else pd.DataFrame()
)

print("firm_status_df:", firm_status_df.shape)
print("window_summary_all_df:", window_summary_all_df.shape)
print("param_draws_all_df:", param_draws_all_df.shape)
print("oos_summary_all_df:", oos_summary_all_df.shape)
print("oos_draws_all_df:", oos_draws_all_df.shape)

display(firm_status_df.head())
display(window_summary_all_df.head())
display(oos_summary_all_df.head())

firm_status_df: (21, 10)
window_summary_all_df: (0, 0)
param_draws_all_df: (0, 0)
oos_summary_all_df: (0, 0)
oos_draws_all_df: (0, 0)


,gvkey,company,ok,elapsed_sec,n_windows_attempted,n_window_rows,n_param_draw_rows,n_oos_summary_rows,n_oos_draw_rows,msg
0,100022,BAYERISCHE MOTOREN WERKE AKT,False,0.027291,36,0,0,0,0,NameError: name 'bayes_merton' is not defined
1,100080,BAYER AG,False,0.024069,36,0,0,0,0,NameError: name 'bayes_merton' is not defined
2,100957,IBERDROLA SA,False,0.023880,36,0,0,0,0,NameError: name 'bayes_merton' is not defined
3,101202,L'AIR LIQUIDE SA,False,0.024392,36,0,0,0,0,NameError: name 'bayes_merton' is not defined
4,101204,SANOFI,False,0.023463,36,0,0,0,0,NameError: name 'bayes_merton' is not defined


""


""


In [ ]:
# main CSV paths
firm_status_path = RUN_DIR / "bayes_merton_firm_status.csv"
window_summary_path = RUN_DIR / "bayes_merton_window_summary.csv"
param_draws_path = RUN_DIR / "bayes_merton_param_draws.csv.gz"
oos_summary_path = RUN_DIR / "bayes_merton_oos_summary.csv"
oos_draws_path = RUN_DIR / "bayes_merton_oos_draws.csv.gz"
windows_used_path = RUN_DIR / "bayes_merton_windows_used.csv"
run_config_path = RUN_DIR / "bayes_merton_run_config.csv"

# save outputs
firm_status_df.to_csv(firm_status_path, index=False)
window_summary_all_df.to_csv(window_summary_path, index=False)
param_draws_all_df.to_csv(param_draws_path, index=False, compression="gzip")
oos_summary_all_df.to_csv(oos_summary_path, index=False)
oos_draws_all_df.to_csv(oos_draws_path, index=False, compression="gzip")

pd.DataFrame(RUN_WINDOWS).to_csv(windows_used_path, index=False)

run_config_df = pd.DataFrame([{
    "run_tag": RUN_TAG,
    "n_firms": len(RUN_GVKEYS),
    "n_windows": len(RUN_WINDOWS),
    "draws": DRAWS,
    "tune": TUNE,
    "chains": CHAINS,
    "cores_inside_pymc": CORES,
    "target_accept": TARGET_ACCEPT,
    "max_treedepth": MAX_TREEDEPTH,
    "delta0": DELTA0,
    "hdi_prob": HDI_PROB,
    "store_oos_draws": STORE_OOS_DRAWS,
    "joblib_n_jobs": N_JOBS,
    "joblib_backend": JOBLIB_BACKEND,
    "run_elapsed_minutes": run_elapsed / 60.0,
}])
run_config_df.to_csv(run_config_path, index=False)

print("Saved files:")
print(firm_status_path)
print(window_summary_path)
print(param_draws_path)
print(oos_summary_path)
print(oos_draws_path)
print(windows_used_path)
print(run_config_path)

In [ ]:
print("Successful firms:", int(firm_status_df["ok"].sum()) if not firm_status_df.empty else 0)
print("Failed firms:", int((~firm_status_df["ok"]).sum()) if not firm_status_df.empty else 0)

if not firm_status_df.empty and (~firm_status_df["ok"]).any():
    display(firm_status_df.loc[~firm_status_df["ok"]].sort_values("gvkey"))

if not window_summary_all_df.empty:
    print("\nWindow-level ok counts:")
    print(window_summary_all_df["ok"].value_counts(dropna=False))

    print("\nMost common window messages:")
    display(window_summary_all_df["msg"].value_counts(dropna=False).head(20))

    cols_diag = [
        "n_divergences", "divergences_pct",
        "mu_rhat", "sigma_rhat",
        "mu_ess_bulk", "sigma_ess_bulk",
    ]
    cols_diag = [c for c in cols_diag if c in window_summary_all_df.columns]
    display(window_summary_all_df[cols_diag].describe())